In [24]:
import json
from collections import OrderedDict

input_file = "../dataset/original/test.json"
output_file = "../dataset/original_formatted/test.json"

with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

assert isinstance(data, list), "최상위가 리스트(JSON array)여야 합니다."
print(f"로드 완료: {len(data)}개 레코드")


로드 완료: 408개 레코드


In [ ]:
from collections import OrderedDict
from collections import Counter

def build_speaker_map(conversation):
    """
    1) 원래 speakerID -> 화자N 매핑(등장 순서)
    2) name1/name2가 누구인지(어느 화자에게 붙는지) 대화 내용으로 추론
       - 규칙: 어떤 화자가 nameX를 언급하면, nameX는 '상대 화자'를 지칭한다고 간주
       - 기본: 화자1→name1, 화자2→name2
    3) speaker_map 형태: {"화자1": {"id": 원래ID, "name": "name1|name2"}, ...}
    4) name_to_alias 도 함께 반환: {"name1": "화자1|화자2", "name2": "화자1|화자2"}
    """
    # 1) 원래 speakerID -> 화자N
    orig_to_alias = OrderedDict()
    idx = 1
    for turn in conversation:
        spk = turn.get("speaker", "")
        if spk not in orig_to_alias:
            orig_to_alias[spk] = f"화자{idx}"
            idx += 1

    # 두 화자 가정(요청 조건)
    aliases = list(OrderedDict.fromkeys(orig_to_alias.values()))
    if len(aliases) < 2:
        # 화자가 1명 이하인 드문 케이스 대비: 기본 이름 부여
        alias_default_names = {"화자1": "name1"}
        name_to_alias = {"name1": "화자1", "name2": "화자2" if len(aliases) == 2 else "화자1"}
        # speaker_map 생성
        speaker_map = OrderedDict()
        for orig_id, alias in orig_to_alias.items():
            speaker_map[alias] = {"id": orig_id, "name": alias_default_names.get(alias, "name2")}
        return orig_to_alias, speaker_map, name_to_alias

    alias1, alias2 = aliases[0], aliases[1]

    # 2) name1/name2가 누구인지 투표
    votes = {"name1": Counter(), "name2": Counter()}
    # 역룩업: alias -> orig_id (각 alias는 유니크한 한 명이라고 가정)
    alias_to_orig = {v: k for k, v in orig_to_alias.items()}

    for turn in conversation:
        spk = turn.get("speaker", "")
        utt = turn.get("utterance", "") or ""
        alias = orig_to_alias.get(spk)
        if alias not in (alias1, alias2):
            continue
        other_alias = alias2 if alias == alias1 else alias1

        # 단순 포함 기준: 'name1'/'name2' 문자열을 발견하면 상대에게 표를 준다.
        if "name1" in utt:
            votes["name1"][other_alias] += utt.count("name1")
        if "name2" in utt:
            votes["name2"][other_alias] += utt.count("name2")

    # 기본 이름 배정
    name_owner = {"name1": alias1, "name2": alias2}

    # 투표 결과 적용
    for nm in ("name1", "name2"):
        if votes[nm]:
            # 가장 많은 득표 alias
            winner_alias, _ = votes[nm].most_common(1)[0]
            name_owner[nm] = winner_alias

    # 충돌 해결: 둘 다 같은 alias로 몰리는 경우
    if name_owner["name1"] == name_owner["name2"]:
        # 더 강한 증거를 가진 쪽을 해당 alias에 두고, 다른 이름은 반대 alias로 보냄
        n1 = votes["name1"][name_owner["name1"]]
        n2 = votes["name2"][name_owner["name2"]]
        if n1 > n2:
            # name1 고정, name2 반대편으로
            name_owner["name2"] = alias2 if name_owner["name1"] == alias1 else alias1
        elif n2 > n1:
            # name2 고정, name1 반대편으로
            name_owner["name1"] = alias2 if name_owner["name2"] == alias1 else alias1
        else:
            # 증거가 동일하면 기본값으로 롤백
            name_owner = {"name1": alias1, "name2": alias2}

    # 3) speaker_map 구성: 각 화자에 {id, name}
    speaker_map = OrderedDict()
    for alias in (alias1, alias2):
        orig_id = alias_to_orig.get(alias, "")
        # 이 alias가 어떤 name인지 역으로 찾아줌
        alias_name = "name1" if name_owner["name1"] == alias else ("name2" if name_owner["name2"] == alias else "")
        speaker_map[alias] = {"id": orig_id, "name": alias_name}

    # 4) name_to_alias 반환(치환에 사용)
    name_to_alias = { "name1": name_owner["name1"], "name2": name_owner["name2"] }

    return orig_to_alias, speaker_map, name_to_alias

def conversation_to_text(conversation, spk_alias, name_to_alias):
    """
    1) 원래 speakerID를 화자N 별칭으로 바꿔 줄 구성
    2) 발화 내용 안의 'name1' / 'name2'를 각각 name_to_alias로 치환
       - 예: "name2님은" -> "화자X님은"
    """
    lines = []
    for turn in conversation:
        spk = turn.get("speaker", "")
        utt = turn.get("utterance", "") or ""
        alias = spk_alias.get(spk, spk)

        # name 토큰을 해당 화자 별칭으로 치환
        # (단순 문자열 치환: 'name1', 'name2'가 토큰으로 쓰이는 전제)
        if "name1" in utt:
            utt = utt.replace("name1", name_to_alias["name1"])
        if "name2" in utt:
            utt = utt.replace("name2", name_to_alias["name2"])

        lines.append(f"{alias}: {utt}")
    return "\n".join(lines)

def transform_record(rec):
    in_obj = rec.get("input", {})
    conversation = in_obj.get("conversation", [])
    subject_keyword = in_obj.get("subject_keyword", [])

    # build_speaker_map이 이제 (spk_to_alias, speaker_map, name_to_alias) 반환
    spk_to_alias, speaker_map, name_to_alias = build_speaker_map(conversation)

    # 대사 내 name1/name2 치환까지 포함
    dialogue_text = conversation_to_text(conversation, spk_to_alias, name_to_alias)

    out = OrderedDict()
    out["id"] = rec.get("id", "")
    out["dialogue"] = dialogue_text
    out["subject_keyword"] = subject_keyword
    out["speaker_map"] = speaker_map
    out["output"] = rec.get("output", "")
    return out

def write_as_one_key_per_line(objs, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write("[\n")
        for i, obj in enumerate(objs):
            f.write("  {\n")
            keys = list(obj.keys())  # OrderedDict로 순서 보장
            for j, k in enumerate(keys):
                v = obj[k]
                v_json = json.dumps(v, ensure_ascii=False)
                comma = "," if j < len(keys) - 1 else ""
                f.write(f'    "{k}": {v_json}{comma}\n')
            f.write("  }")
            if i < len(objs) - 1:
                f.write(",")
            f.write("\n")
        f.write("]\n")


In [26]:
from tqdm import tqdm
# 8) tqdm 진행률 표시하며 전체 변환 실행
transformed = [transform_record(rec) for rec in tqdm(data, desc="Transforming", unit="record")]
print(f"변환 완료: {len(transformed)}개 레코드")


Transforming: 100%|██████████| 408/408 [00:00<00:00, 52974.12record/s]

변환 완료: 408개 레코드


In [27]:
# 9) 저장
write_as_one_key_per_line(transformed, output_file)
print(f"저장 완료: {output_file}")


저장 완료: ../dataset/original_formatted/test.json
